# Highland Square Conjoint — Notebook 1: Design & Run

**Goal:** Build a Choice-Based Conjoint (CBC) experiment for Highland Square Apartments, generate a D-efficient design, validate against realism rules, and run the experiment through Claude as the simulated respondent.

**Pipeline (this notebook):**
1. Define attributes & levels (locked from prior conversation)
2. Define realism rules / prohibited combinations
3. Generate full factorial → filter to valid profiles
4. Build a balanced D-efficient set of choice tasks
5. Define 3 renter personas
6. Render a sample HTML preview for stakeholder review
7. Pilot run (small) → full run (async, parallel)
8. Save raw responses to CSV

**Output:** `data/raw_responses.csv` ready for Notebook 2 (analysis).

## 0. Setup

In [ ]:
# Install only what's not already on your machine. Comment out if already installed.
# !pip install anthropic pandas numpy --quiet

In [1]:
import os
import json
import random
import asyncio
import itertools
import time
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import anthropic

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Output directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Setup complete.")

Setup complete.


In [ ]:
# API key — set this as an environment variable, not hardcoded.
# In your terminal before launching jupyter:
#   export 
# Or set it inline here for convenience:
os.environ["ANTHROPIC_API_KEY"] = "NA"

assert os.environ.get("ANTHROPIC_API_KEY"), 
)
print("API key found.")

API key found.


## 1. Attribute & Level Specification

Locked from our prior discussion. 8 attributes, 24 levels total.

**Two-component price coding (Option C):** Size and Price both vary as continuous attributes so we can decompose into per-SF utility and amenity premium during estimation.

In [3]:
# Each attribute: dict of {level_label: numeric_value_for_continuous_attrs_or_None}
# Continuous attrs (Size, Price) carry numeric values for the regression.
# Categorical attrs carry None and will be dummy-coded later.

ATTRIBUTES = {
    # ----- Pricing & lease economics -----
    "Size": {
        "750 SF (compact 1BR)": 750,
        "1,000 SF (large 1BR / compact 2BR)": 1000,
        "1,250 SF (standard 2BR)": 1250,
    },
    "Price": {
        "$1,650/mo": 1650,
        "$1,950/mo": 1950,
        "$2,250/mo": 2250,
        "$2,550/mo": 2550,
    },
    "MoveInSpecial": {
        "None": None,
        "1 month free (12-mo lease)": None,
        "2 months free (13-mo lease)": None,
    },

    # ----- Location & micro-location -----
    "Location": {
        "North Druid Hills / Briarcliff": None,
        "Virginia-Highland / Morningside": None,
        "Decatur / Emory Village": None,
        "Buckhead / Lenox": None,
    },
    "CommuteToWork": {
    "Quick (under 15 min by car)": None,
    "Average (15-30 min by car)": None,
    "Long (over 30 min by car)": None,
    },
    "Walkability": {
        "Car-Required (drive for groceries, dining, errands from this building)": None,
        "Walkable Errands (groceries & a few restaurants within a 10-min walk of this building)": None,
        "Walk Everywhere (daily errands, dining, transit within a 10-min walk of this building)": None,
    },

    # ----- In-unit & building -----
    "Finishes": {
        "Builder-grade (laminate counters, basic appliances, no smart features)": None,
        "Mid-tier (granite/quartz counters, stainless appliances, in-unit washer/dryer)": None,
        "Premium (quartz waterfall island, smart thermostat, keyless entry, video doorbell)": None,
    },
    "Parking": {
        "Surface lot, unassigned parking, no gate": None,
        "Gated surface lot + reserved space option": None,
        "Dedicated garage with assigned space + EV charging": None,
    },
    "Security": {
        "Tier 1: Open access, no controlled entry": None,
        "Tier 2: Perimeter gate + controlled-access lobby + camera coverage": None,
        "Tier 3: Tier 2 + 24/7 staff or virtual concierge + smart locks throughout": None,
    },

    # ----- Specific amenity binaries -----
    "Rooftop": {
        "No rooftop space": None,
        "Rooftop lounge with skyline views & outdoor seating": None,
    },
    "Coworking": {
        "No dedicated coworking space": None,
        "Resident co-working lounge with private call rooms & wifi": None,
    },
    "PetAmenities": {
        "No pet amenities": None,
        "Standard dog park only": None,
        "Dog park + pet spa with grooming station": None,
    },
    "PackageHandling": {
        "Standard mailroom (sign for packages during office hours)": None,
        "24/7 Amazon Hub lockers + refrigerated grocery locker": None,
    },
}

# Quick summary
for attr, levels in ATTRIBUTES.items():
    print(f"{attr:18s} | {len(levels)} levels")

total_full_factorial = np.prod([len(v) for v in ATTRIBUTES.values()])
print(f"\nTotal attributes: {len(ATTRIBUTES)}")
print(f"Full factorial profile space: {total_full_factorial:,} profiles")

Size               | 3 levels
Price              | 4 levels
MoveInSpecial      | 3 levels
Location           | 4 levels
CommuteToWork      | 3 levels
Walkability        | 3 levels
Finishes           | 3 levels
Parking            | 3 levels
Security           | 3 levels
Rooftop            | 2 levels
Coworking          | 2 levels
PetAmenities       | 3 levels
PackageHandling    | 2 levels

Total attributes: 13
Full factorial profile space: 839,808 profiles


## 2. Realism Rules / Prohibited Combinations

Six rules from the locked spec, plus we use the price-per-SF rule as continuous logic rather than discrete pairs.

In [4]:
def is_valid_profile(profile: dict) -> bool:
    sec = profile["Security"]
    fin = profile["Finishes"]
    park = profile["Parking"]
    rooftop = profile["Rooftop"]
    sf = ATTRIBUTES["Size"][profile["Size"]]
    price = ATTRIBUTES["Price"][profile["Price"]]

    # Rule 1: Price/SF realism band
    psf = price / sf
    if psf < 1.40 or psf > 3.20:
        return False

    # Rule 2: Premium smart finishes imply at least Tier 2 access control
    if fin.startswith("Premium") and sec.startswith("Tier 1"):
        return False

    # Rule 3: Tier 3 security implies at least gated parking
    if sec.startswith("Tier 3") and park.startswith("Surface lot, unassigned"):
        return False

    # Rule 4: Rooftop lounges in Class A always have keycard access
    if rooftop.startswith("Rooftop lounge") and sec.startswith("Tier 1"):
        return False

    return True

In [5]:
# Generate full factorial and filter
attr_names = list(ATTRIBUTES.keys())
level_lists = [list(ATTRIBUTES[a].keys()) for a in attr_names]

all_profiles = []
for combo in itertools.product(*level_lists):
    profile = dict(zip(attr_names, combo))
    if is_valid_profile(profile):
        all_profiles.append(profile)

print(f"Full factorial:    {total_full_factorial:,} profiles")
print(f"Valid (realistic): {len(all_profiles):,} profiles ({100*len(all_profiles)/total_full_factorial:.1f}% retained)")

Full factorial:    839,808 profiles
Valid (realistic): 466,560 profiles (55.6% retained)


In [6]:
# Sanity-check level balance across the valid pool
valid_df = pd.DataFrame(all_profiles)
print("Level frequency in valid pool:\n")
for attr in attr_names:
    counts = valid_df[attr].value_counts()
    pct = (counts / len(valid_df) * 100).round(1)
    print(f"--- {attr} ---")
    for level, p in pct.items():
        print(f"  {p:5.1f}%  {level[:70]}")
    print()

Level frequency in valid pool:

--- Size ---
   40.0%  1,000 SF (large 1BR / compact 2BR)
   30.0%  750 SF (compact 1BR)
   30.0%  1,250 SF (standard 2BR)

--- Price ---
   30.0%  $1,950/mo
   30.0%  $2,250/mo
   20.0%  $1,650/mo
   20.0%  $2,550/mo

--- MoveInSpecial ---
   33.3%  None
   33.3%  1 month free (12-mo lease)
   33.3%  2 months free (13-mo lease)

--- Location ---
   25.0%  North Druid Hills / Briarcliff
   25.0%  Virginia-Highland / Morningside
   25.0%  Decatur / Emory Village
   25.0%  Buckhead / Lenox

--- CommuteToWork ---
   33.3%  Quick (under 15 min by car)
   33.3%  Average (15-30 min by car)
   33.3%  Long (over 30 min by car)

--- Walkability ---
   33.3%  Car-Required (drive for groceries, dining, errands from this building)
   33.3%  Walkable Errands (groceries & a few restaurants within a 10-min walk o
   33.3%  Walk Everywhere (daily errands, dining, transit within a 10-min walk o

--- Finishes ---
   36.1%  Builder-grade (laminate counters, basic appliance

## 3. Build a D-Efficient Choice Task Set

Rather than random pair sampling (which left your senior-housing model with `nan` standard errors), we build a **balanced fractional design** using a level-balanced random search with D-efficiency scoring. This is a pragmatic alternative to formal D-optimal algorithms (which require `pyDOE2`/`OApackage`) and works well for studies of this size.

**Design parameters:**
- 24 choice tasks per respondent
- 3 alternatives per task (no "None" option — we'll add an outside option in Notebook 2 if needed)
- We score candidate designs by level-balance (each level appears ~equally often) and pick the best of 200 random candidates.

In [7]:
N_TASKS = 36
N_ALTS = 3
N_RANDOM_STARTS = 50            # number of independent random starting designs
N_SWAPS_PER_START = 2000        # swap iterations per starting design

def build_random_design(profile_pool, n_tasks, n_alts, rng):
    """Sample n_tasks * n_alts profiles without replacement (within each task)."""
    return [rng.sample(profile_pool, n_alts) for _ in range(n_tasks)]

def level_balance_score(design, attr_names, level_lists):
    """Sum of squared deviations from uniform within each attribute. Lower = better."""
    flat = [alt for task in design for alt in task]
    total_score = 0
    for i, attr in enumerate(attr_names):
        levels = level_lists[i]
        target = len(flat) / len(levels)
        observed = {lvl: 0 for lvl in levels}
        for prof in flat:
            observed[prof[attr]] += 1
        total_score += sum((c - target) ** 2 for c in observed.values())
    return total_score

def greedy_swap_optimize(design, profile_pool, attr_names, level_lists, n_swaps, rng):
    """Iteratively swap a single alternative with a random profile from the pool.
    Keep the swap if it lowers the score; otherwise revert."""
    current_score = level_balance_score(design, attr_names, level_lists)
    n_tasks = len(design)
    n_alts = len(design[0])

    for _ in range(n_swaps):
        # Pick a random alternative position
        task_idx = rng.randrange(n_tasks)
        alt_idx = rng.randrange(n_alts)
        # Try swapping it with a random profile from the pool
        candidate = rng.choice(profile_pool)
        # Skip if candidate already in this task (avoid duplicate alternatives)
        if any(candidate == design[task_idx][k] for k in range(n_alts) if k != alt_idx):
            continue
        original = design[task_idx][alt_idx]
        design[task_idx][alt_idx] = candidate
        new_score = level_balance_score(design, attr_names, level_lists)
        if new_score < current_score:
            current_score = new_score
        else:
            design[task_idx][alt_idx] = original  # revert
    return design, current_score

# Run the optimization
rng = random.Random(RANDOM_SEED)
best_design = None
best_score = float("inf")

for start in range(N_RANDOM_STARTS):
    candidate = build_random_design(all_profiles, N_TASKS, N_ALTS, rng)
    candidate, score = greedy_swap_optimize(
        candidate, all_profiles, attr_names, level_lists, N_SWAPS_PER_START, rng
    )
    if score < best_score:
        best_score = score
        best_design = candidate
    if (start + 1) % 10 == 0:
        print(f"  Restart {start + 1}/{N_RANDOM_STARTS} | Best score so far: {best_score:.1f}")

print(f"\nFinal best level-balance score: {best_score:.1f}")
print(f"Total alternatives in design: {N_TASKS * N_ALTS}")

  Restart 10/50 | Best score so far: 10.0
  Restart 20/50 | Best score so far: 10.0
  Restart 30/50 | Best score so far: 10.0
  Restart 40/50 | Best score so far: 10.0
  Restart 50/50 | Best score so far: 10.0

Final best level-balance score: 10.0
Total alternatives in design: 108


In [8]:
# Verify level balance in the chosen design
flat_alts = [alt for task in best_design for alt in task]
design_df = pd.DataFrame(flat_alts)
print(f"Design contains {len(flat_alts)} alternatives across {N_TASKS} tasks.\n")
for attr in attr_names:
    print(f"--- {attr} ---")
    counts = design_df[attr].value_counts()
    for lvl, c in counts.items():
        pct = c / len(flat_alts) * 100
        target_pct = 100 / len(ATTRIBUTES[attr])
        flag = " ⚠" if abs(pct - target_pct) > 8 else ""
        print(f"  {c:3d} ({pct:5.1f}% vs target {target_pct:.1f}%){flag}  {lvl[:60]}")
    print()

Design contains 108 alternatives across 36 tasks.

--- Size ---
   37 ( 34.3% vs target 33.3%)  1,250 SF (standard 2BR)
   36 ( 33.3% vs target 33.3%)  750 SF (compact 1BR)
   35 ( 32.4% vs target 33.3%)  1,000 SF (large 1BR / compact 2BR)

--- Price ---
   27 ( 25.0% vs target 25.0%)  $2,250/mo
   27 ( 25.0% vs target 25.0%)  $2,550/mo
   27 ( 25.0% vs target 25.0%)  $1,650/mo
   27 ( 25.0% vs target 25.0%)  $1,950/mo

--- MoveInSpecial ---
   37 ( 34.3% vs target 33.3%)  1 month free (12-mo lease)
   36 ( 33.3% vs target 33.3%)  None
   35 ( 32.4% vs target 33.3%)  2 months free (13-mo lease)

--- Location ---
   28 ( 25.9% vs target 25.0%)  Virginia-Highland / Morningside
   27 ( 25.0% vs target 25.0%)  Decatur / Emory Village
   27 ( 25.0% vs target 25.0%)  Buckhead / Lenox
   26 ( 24.1% vs target 25.0%)  North Druid Hills / Briarcliff

--- CommuteToWork ---
   36 ( 33.3% vs target 33.3%)  Quick (under 15 min by car)
   36 ( 33.3% vs target 33.3%)  Average (15-30 min by car)
   36 

In [9]:
# Save the design for reference
design_records = []
for task_id, task in enumerate(best_design):
    for alt_id, profile in enumerate(task):
        rec = {"Task_ID": task_id, "Alt_ID": alt_id}
        rec.update(profile)
        design_records.append(rec)

design_save = pd.DataFrame(design_records)
design_save.to_csv(DATA_DIR / "choice_tasks.csv", index=False)
print(f"Saved choice design to {DATA_DIR / 'choice_tasks.csv'}")
design_save.head(6)

Saved choice design to data/choice_tasks.csv


,Task_ID,Alt_ID,Size,Price,MoveInSpecial,Location,CommuteToWork,Walkability,Finishes,Parking,Security,Rooftop,Coworking,PetAmenities,PackageHandling
0,0,0,"1,250 SF (standard 2BR)","$2,250/mo",None,Virginia-Highland / Morningside,Quick (under 15 min by car),"Car-Required (drive for groceries, dining, err...","Mid-tier (granite/quartz counters, stainless a...",Gated surface lot + reserved space option,"Tier 1: Open access, no controlled entry",No rooftop space,No dedicated coworking space,Standard dog park only,24/7 Amazon Hub lockers + refrigerated grocery...
1,0,1,"1,000 SF (large 1BR / compact 2BR)","$2,550/mo",2 months free (13-mo lease),Decatur / Emory Village,Average (15-30 min by car),Walkable Errands (groceries & a few restaurant...,"Mid-tier (granite/quartz counters, stainless a...",Dedicated garage with assigned space + EV char...,Tier 3: Tier 2 + 24/7 staff or virtual concier...,Rooftop lounge with skyline views & outdoor se...,No dedicated coworking space,Dog park + pet spa with grooming station,24/7 Amazon Hub lockers + refrigerated grocery...
2,0,2,"1,000 SF (large 1BR / compact 2BR)","$2,550/mo",1 month free (12-mo lease),Decatur / Emory Village,Long (over 30 min by car),"Walk Everywhere (daily errands, dining, transi...","Mid-tier (granite/quartz counters, stainless a...","Surface lot, unassigned parking, no gate",Tier 2: Perimeter gate + controlled-access lob...,Rooftop lounge with skyline views & outdoor se...,Resident co-working lounge with private call r...,No pet amenities,Standard mailroom (sign for packages during of...
3,1,0,"1,000 SF (large 1BR / compact 2BR)","$2,550/mo",1 month free (12-mo lease),Decatur / Emory Village,Quick (under 15 min by car),Walkable Errands (groceries & a few restaurant...,"Premium (quartz waterfall island, smart thermo...",Gated surface lot + reserved space option,Tier 3: Tier 2 + 24/7 staff or virtual concier...,Rooftop lounge with skyline views & outdoor se...,No dedicated coworking space,Standard dog park only,24/7 Amazon Hub lockers + refrigerated grocery...
4,1,1,"1,250 SF (standard 2BR)","$2,250/mo",2 months free (13-mo lease),Virginia-Highland / Morningside,Long (over 30 min by car),Walkable Errands (groceries & a few restaurant...,"Builder-grade (laminate counters, basic applia...",Dedicated garage with assigned space + EV char...,"Tier 1: Open access, no controlled entry",No rooftop space,Resident co-working lounge with private call r...,Dog park + pet spa with grooming station,Standard mailroom (sign for packages during of...
5,1,2,750 SF (compact 1BR),"$1,650/mo",1 month free (12-mo lease),Buckhead / Lenox,Long (over 30 min by car),Walkable Errands (groceries & a few restaurant...,"Builder-grade (laminate counters, basic applia...",Gated surface lot + reserved space option,Tier 2: Perimeter gate + controlled-access lob...,Rooftop lounge with skyline views & outdoor se...,No dedicated coworking space,Dog park + pet spa with grooming station,24/7 Amazon Hub lockers + refrigerated grocery...


## 4. Renter Personas

Three personas matched to Highland Square's actual renter mix:
1. **Emory grad student / postdoc** — price-sensitive, walkability-tilted
2. **Mid-career VaHi/Morningside professional** — finish & lifestyle-tilted, less price-sensitive
3. **Empty-nester downsizer** — security & parking-tilted, location-loyal

We'll run the full design through each persona and either pool with persona dummies or estimate per-cohort models in Notebook 2.

In [10]:
PERSONAS = {
    "emory_grad": """You are Maya, 28. You are a third-year PhD candidate at Emory University.
Your stipend is $42,000/year plus $10,000/year in research assistantship funding. You have $580/month in federal student loan payments. You have $4,200 in liquid savings and no other significant assets.
You moved to Atlanta from Wisconsin two years ago.
Your daily destination is Emory's main campus. You expect to live in this apartment for at least 18 months, through the end of your dissertation.""",

    "vahi_professional": """You are David, 34. You are a senior consultant at a healthcare strategy firm.
Your base salary is $135,000/year with a typical annual bonus of $20,000. You have $42,000 in a high-yield savings account and a 401(k) balance of $185,000. You have no debt other than student loans paid down to $14,000. You moved from Boston eight months ago.
Your daily destination is your firm's office on West Peachtree in Midtown three days a week; you work from home the other two days. You expect to live in this apartment for 2–3 years.""",

    "empty_nester": """You are Patricia, 58. You recently retired from a 25-year career as a hospital administrator. Your husband Tom, 60, still works as a CPA.
Combined household income is $180,000/year. You sold your 4-bedroom Sandy Springs home last September for $815,000 (purchased in 1998 for $310,000). You and Tom have approximately $1.4M in retirement and brokerage accounts.
You and Tom are renting for the first time in 30 years. You have no children at home — your two adult children live in Charlotte and Denver.
You expect to live in this apartment for 2–4 years while deciding whether to purchase a smaller condo.""",

    "skeptical_renter_control": """You are Alex, 31. You are a software engineer at a mid-sized fintech company in Atlanta.
Your salary is $115,000/year. You have $35,000 in liquid savings, a 401(k) balance of $95,000, and no significant debt. You have lived in Atlanta for four years and have rented in Midtown, West Midtown, and Decatur previously. You expect to live in this apartment for 1–2 years.""",
}

for name, prompt in PERSONAS.items():
    print(f"=== {name} ===")
    print(prompt[:200] + "...\n")

=== emory_grad ===
You are Maya, 28. You are a third-year PhD candidate at Emory University.
Your stipend is $42,000/year plus $10,000/year in research assistantship funding. You have $580/month in federal student loan ...

=== vahi_professional ===
You are David, 34. You are a senior consultant at a healthcare strategy firm.
Your base salary is $135,000/year with a typical annual bonus of $20,000. You have $42,000 in a high-yield savings account...

=== empty_nester ===
You are Patricia, 58. You recently retired from a 25-year career as a hospital administrator. Your husband Tom, 60, still works as a CPA.
Combined household income is $180,000/year. You sold your 4-be...

=== skeptical_renter_control ===
You are Alex, 31. You are a software engineer at a mid-sized fintech company in Atlanta.
Your salary is $115,000/year. You have $35,000 in liquid savings, a 401(k) balance of $95,000, and no significa...



## 5. HTML Preview (Optional — for stakeholder review)

Render the first 5 choice tasks as static HTML cards so you can sanity-check what Claude will see.

In [11]:
import html

def render_profile(profile):
    rows = ""
    for attr in attr_names:
        rows += f"<li><strong>{attr}:</strong> {html.escape(str(profile[attr]))}</li>"
    return f"<ul style='padding-left:18px;font-size:13px'>{rows}</ul>"

preview_html = ["<html><head><style>body{font-family:Arial;background:#f5f7fb;padding:20px}.task{background:white;border:1px solid #ddd;border-radius:8px;padding:14px;margin-bottom:14px}.opts{display:grid;grid-template-columns:repeat(3,1fr);gap:10px}.opt{border:1px solid #eee;border-radius:6px;padding:10px;background:#fafbff}h2{color:#1d4ed8;font-size:16px}h3{font-size:14px;margin:0 0 6px}</style></head><body>"]
preview_html.append("<h1>Highland Square Conjoint — First 5 Choice Tasks</h1>")

for i, task in enumerate(best_design[:5], 1):
    preview_html.append(f"<div class='task'><h2>Choice Task {i} of {N_TASKS}</h2>")
    preview_html.append("<p><em>Imagine you are searching for an apartment. Which of these would you most likely choose?</em></p>")
    preview_html.append("<div class='opts'>")
    for j, profile in enumerate(task):
        label = chr(ord('A') + j)
        preview_html.append(f"<div class='opt'><h3>Option {label}</h3>{render_profile(profile)}</div>")
    preview_html.append("</div></div>")

preview_html.append("</body></html>")
preview_path = DATA_DIR / "choice_task_preview.html"
preview_path.write_text("".join(preview_html), encoding="utf-8")
print(f"Saved preview: {preview_path.resolve()}")
print("Open this file in a browser to review the choice cards.")

Saved preview: /Users/elyas/vscode/tenant-choice-modeling/notebooks/data/choice_task_preview.html
Open this file in a browser to review the choice cards.


## 6. Pilot Run (Small — Validates the Pipeline)

Run 3 tasks × 2 personas × 2 reps = 12 calls to confirm everything works before the full run.

In [15]:
def format_choice_prompt(task, attr_names):
    """Format a single choice task as a user prompt."""
    lines = [
        "Imagine you are searching for an apartment in Atlanta. Below are three options "
        "to evaluate, plus the option to keep your current housing situation.\n",
        "Note: 'Location' is the broader submarket. 'Walkability' describes the immediate block "
        "around the building (every Atlanta submarket has both walkable enclaves and "
        "car-dependent stretches, so the two attributes can vary independently).\n",
    ]
    for j, profile in enumerate(task):
        label = chr(ord('A') + j)
        lines.append(f"Option {label}:")
        for attr in attr_names:
            lines.append(f"  - {attr}: {profile[attr]}")
        lines.append("")
    
    # Add the outside option
    lines.append("Option D:")
    lines.append("  - Stay in my current apartment / keep searching — none of these are right for me")
    lines.append("")
    lines.append("Which option would you choose? Reply with just the letter (A, B, C, or D) followed by one short sentence explaining why.")
    return "\n".join(lines)

In [73]:
# Sync pilot — sequential, easy to debug
client = anthropic.Anthropic()

MODEL = "claude-haiku-4-5-20251001"  # fast & cheap for pilot; switch to sonnet for full run
PILOT_TASKS = best_design[:3]
PILOT_PERSONAS = list(PERSONAS.keys())[:2]
PILOT_REPS = 2

pilot_results = []
rng_pilot = random.Random(RANDOM_SEED)

for task_id, task in enumerate(PILOT_TASKS):
    for persona_name in PILOT_PERSONAS:
        for rep in range(PILOT_REPS):
            shuffled = list(task)
            rng_pilot.shuffle(shuffled)
            user_prompt = format_choice_prompt(shuffled, attr_names)
            try:
                response = client.messages.create(
                    model=MODEL,
                    max_tokens=120,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": user_prompt}],
                )
                text = response.content[0].text
                pilot_results.append({
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Shuffled_Profiles": json.dumps(shuffled),
                    "Response": text,
                })
                print(f"Task {task_id} | {persona_name} | rep {rep} → {text[:80]}")
            except Exception as e:
                print(f"Error: {e}")
                time.sleep(2)

print(f"\nPilot complete: {len(pilot_results)} responses.")

Task 0 | emory_grad | rep 0 → B

The $300/month savings compared to A and C is meaningful on my stipend, the s
Task 0 | emory_grad | rep 1 → C

It's the cheapest option ($2,250/mo vs $2,550), has the most space for a simi
Task 0 | vahi_professional | rep 0 → C

Quick commute to Midtown (15 min vs 30+ for others), lowest rent ($2,250), la
Task 0 | vahi_professional | rep 1 → C

Quick commute to Midtown (15 min vs 30+ min saves ~5 hours/week), largest spa
Task 1 | emory_grad | rep 0 → B

The 15-minute commute to Emory justifies the higher rent given my stipend, an
Task 1 | emory_grad | rep 1 → A

The 15-minute commute and walkable neighborhood justify the premium price giv
Task 1 | vahi_professional | rep 0 → C

The 15-minute commute to Midtown saves 30+ hours annually versus the other op
Task 1 | vahi_professional | rep 1 → C

The 15-minute commute to Midtown saves 30+ hours yearly, the premium finishes
Task 2 | emory_grad | rep 0 → B

At $2,250/mo with a quick commute, walkable neighbo

In [74]:
# Quick check: are responses parseable?
import re
def parse_choice(text):
    m = re.search(r"\b([ABC])\b", str(text)[:100])
    return m.group(1) if m else None

pilot_df = pd.DataFrame(pilot_results)
pilot_df["Choice"] = pilot_df["Response"].apply(parse_choice)
print(f"Parseable: {pilot_df['Choice'].notna().sum()} / {len(pilot_df)}")
pilot_df[["Task_ID", "Persona", "Rep", "Choice"]].head(12)

Parseable: 12 / 12


,Task_ID,Persona,Rep,Choice
0,0,emory_grad,0,B
1,0,emory_grad,1,C
2,0,vahi_professional,0,C
3,0,vahi_professional,1,C
4,1,emory_grad,0,B
5,1,emory_grad,1,A
6,1,vahi_professional,0,C
7,1,vahi_professional,1,C
8,2,emory_grad,0,B
9,2,emory_grad,1,B


In [75]:
# === VALIDATION PILOT: 2 tasks × 4 personas × 3 reps = 24 calls ===
# Confirms Patricia and Alex behave reasonably before the full run.

VALIDATION_MODEL = "claude-haiku-4-5-20251001"
VALIDATION_TASKS = best_design[:2]                  # first 2 tasks
VALIDATION_PERSONAS = list(PERSONAS.keys())         # all 4 personas
VALIDATION_REPS = 3                                 # 3 reps each

client = anthropic.Anthropic()
validation_results = []
rng_val = random.Random(RANDOM_SEED)

print(f"Running {len(VALIDATION_TASKS) * len(VALIDATION_PERSONAS) * VALIDATION_REPS} validation calls...\n")

for task_id, task in enumerate(VALIDATION_TASKS):
    for persona_name in VALIDATION_PERSONAS:
        for rep in range(VALIDATION_REPS):
            shuffled = list(task)
            rng_val.shuffle(shuffled)
            user_prompt = format_choice_prompt(shuffled, attr_names)
            try:
                response = client.messages.create(
                    model=VALIDATION_MODEL,
                    max_tokens=120,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": user_prompt}],
                )
                text = response.content[0].text
                validation_results.append({
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Response": text,
                })
                print(f"Task {task_id} | {persona_name:25s} | rep {rep} → {text[:100]}")
            except Exception as e:
                print(f"Error on Task {task_id} | {persona_name} | rep {rep}: {e}")
                time.sleep(2)

print(f"\nValidation complete: {len(validation_results)} responses.\n")

# Quick D-rate diagnostic per persona
import re
def parse_choice(text):
    m = re.search(r"\b([ABCD])\b", str(text)[:100])
    return m.group(1) if m else None

val_df = pd.DataFrame(validation_results)
val_df["Choice"] = val_df["Response"].apply(parse_choice)

print("=== Choice distribution per persona ===")
for persona in VALIDATION_PERSONAS:
    sub = val_df[val_df["Persona"] == persona]
    counts = sub["Choice"].value_counts().to_dict()
    print(f"{persona:25s} | {counts}")

Running 24 validation calls...

Task 0 | emory_grad                | rep 0 → B

The $300/month savings versus Option A, combined with a shorter commute and larger space, outweig
Task 0 | emory_grad                | rep 1 → C. It's the most affordable option with the shortest commute, leaving me more monthly cushion for my
Task 0 | emory_grad                | rep 2 → C

The $300/month savings ($3,600/year) is meaningful on my stipend, the quick commute minimizes tim
Task 0 | vahi_professional         | rep 0 → C

Quick commute to Midtown saves time and gas money, the $300/month savings ($3,600/year) meaningfu
Task 0 | vahi_professional         | rep 1 → B

It offers the best value ($2,250/mo, largest space) with a quick commute to my main office, and w
Task 0 | vahi_professional         | rep 2 → **B**

Best value for my situation: shortest feasible commute to Midtown, walkable neighborhood for 
Task 0 | empty_nester              | rep 0 → A. The combination of reasonable commute, walka

## 7. Full Async Run

Now we run the full experiment in parallel using `AsyncAnthropic`. With 24 tasks × 3 personas × 30 reps = **2,160 calls**, batched at concurrency=10, this finishes in ~5–10 minutes vs. ~45 minutes sequentially.

**Tunables:**
- `REPETITIONS_PER_PERSONA`: simulated respondents per persona (30 is a reasonable starting point)
- `CONCURRENCY`: max parallel API calls — keep ≤10 to respect rate limits
- `MODEL`: `claude-haiku-4-5-20251001` (fast/cheap) or `claude-sonnet-4-5` (more nuanced reasoning)

In [12]:
# === FULL EXPERIMENT RUN ===

FULL_MODEL = "claude-haiku-4-5-20251001"  # change to sonnet for higher fidelity
REPETITIONS_PER_PERSONA = 50
CONCURRENCY = 10

total_calls = N_TASKS * len(PERSONAS) * REPETITIONS_PER_PERSONA
print(f"Full run will make {total_calls:,} API calls.")
print(f"At concurrency={CONCURRENCY}, est. wall time: {total_calls / CONCURRENCY * 1.5 / 60:.1f} minutes")



Full run will make 7,200 API calls.
At concurrency=10, est. wall time: 18.0 minutes


In [77]:
async def run_one_call(client, semaphore, task_id, task, persona_name, rep, attr_names, rng_seed):
    """Single async API call, with retries on transient errors."""
    rng_local = random.Random(rng_seed)
    shuffled = list(task)
    rng_local.shuffle(shuffled)
    prompt = format_choice_prompt(shuffled, attr_names)

    async with semaphore:
        for attempt in range(3):
            try:
                response = await client.messages.create(
                    model=FULL_MODEL,
                    max_tokens=120,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": prompt}],
                )
                return {
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Shuffled_Profiles": json.dumps(shuffled),
                    "Response": response.content[0].text,
                    "Error": None,
                }
            except Exception as e:
                if attempt == 2:
                    return {
                        "Task_ID": task_id,
                        "Persona": persona_name,
                        "Rep": rep,
                        "Shuffled_Profiles": json.dumps(shuffled),
                        "Response": None,
                        "Error": str(e),
                    }
                await asyncio.sleep(2 ** attempt)
        # Defensive fallback (should never reach here)
        return {
            "Task_ID": task_id,
            "Persona": persona_name,
            "Rep": rep,
            "Shuffled_Profiles": json.dumps(shuffled),
            "Response": None,
            "Error": "Loop exited unexpectedly",
        }

async def run_full_experiment():
    async_client = anthropic.AsyncAnthropic()
    semaphore = asyncio.Semaphore(CONCURRENCY)

    tasks_coros = []
    call_idx = 0
    for task_id, task in enumerate(best_design):
        for persona_name in PERSONAS:
            for rep in range(REPETITIONS_PER_PERSONA):
                seed = RANDOM_SEED + call_idx
                tasks_coros.append(
                    run_one_call(async_client, semaphore, task_id, task, persona_name, rep, attr_names, seed)
                )
                call_idx += 1

    print(f"Dispatching {len(tasks_coros)} async calls (concurrency={CONCURRENCY})...")
    start = time.time()

    # Run with progress reporting AND incremental checkpointing
    results = []
    completed = 0
    chunk = 200  # save every 200 results
    
    for i in range(0, len(tasks_coros), chunk):
        batch = tasks_coros[i:i+chunk]
        batch_results = await asyncio.gather(*batch)
        results.extend(batch_results)
        completed += len(batch_results)
        elapsed = time.time() - start
        rate = completed / elapsed if elapsed > 0 else 0
        
        # Incremental save — overwrites each chunk so you always have the latest
        pd.DataFrame(results).to_csv(DATA_DIR / "raw_responses_partial.csv", index=False)
        
        print(f"  {completed}/{len(tasks_coros)} done ({rate:.1f}/s, elapsed {elapsed:.0f}s) — checkpoint saved")

    return results

# Run it
all_results = await run_full_experiment()
print(f"\nDone. {len(all_results)} responses collected.")

Dispatching 7200 async calls (concurrency=10)...
  200/7200 done (1.5/s, elapsed 130s) — checkpoint saved
  400/7200 done (1.4/s, elapsed 288s) — checkpoint saved
  600/7200 done (1.3/s, elapsed 451s) — checkpoint saved
  800/7200 done (1.3/s, elapsed 612s) — checkpoint saved
  1000/7200 done (1.3/s, elapsed 768s) — checkpoint saved
  1200/7200 done (1.3/s, elapsed 925s) — checkpoint saved
  1400/7200 done (1.3/s, elapsed 1089s) — checkpoint saved
  1600/7200 done (1.3/s, elapsed 1248s) — checkpoint saved
  1800/7200 done (1.3/s, elapsed 1414s) — checkpoint saved
  2000/7200 done (1.3/s, elapsed 1575s) — checkpoint saved
  2200/7200 done (1.3/s, elapsed 1735s) — checkpoint saved
  2400/7200 done (1.3/s, elapsed 1891s) — checkpoint saved
  2600/7200 done (1.3/s, elapsed 2053s) — checkpoint saved
  2800/7200 done (1.3/s, elapsed 2217s) — checkpoint saved
  3000/7200 done (1.3/s, elapsed 2378s) — checkpoint saved
  3200/7200 done (1.3/s, elapsed 2545s) — checkpoint saved
  3400/7200 done 

In [78]:
# Save raw results
results_df = pd.DataFrame(all_results)
errors = results_df["Error"].notna().sum()
print(f"Errors: {errors} / {len(results_df)} ({100*errors/len(results_df):.1f}%)")

out_path = DATA_DIR / "raw_responses.csv"
results_df.to_csv(out_path, index=False)
print(f"\nSaved raw responses to {out_path.resolve()}")
results_df.head()

Errors: 2516 / 7200 (34.9%)

Saved raw responses to /Users/elyas/vscode/tenant-choice-modeling/notebooks/data/raw_responses.csv


,Task_ID,Persona,Rep,Shuffled_Profiles,Response,Error
0,0,emory_grad,0,"[{""Size"": ""1,000 SF (large 1BR / compact 2BR)""...","B\n\nAs a PhD candidate with tight finances, t...",None
1,0,emory_grad,1,"[{""Size"": ""1,000 SF (large 1BR / compact 2BR)""...","C. The $300/month savings ($3,600/year) is mea...",None
2,0,emory_grad,2,"[{""Size"": ""1,000 SF (large 1BR / compact 2BR)""...",B\n\nMy short commute to Emory (under 15 min) ...,None
3,0,emory_grad,3,"[{""Size"": ""1,250 SF (standard 2BR)"", ""Price"": ...","C\n\nThe effective monthly cost ($1,962 after ...",None
4,0,emory_grad,4,"[{""Size"": ""1,000 SF (large 1BR / compact 2BR)""...",C. It's the only option that leaves me with en...,None


In [14]:
# Diagnose the error pattern
import pandas as pd

results_df = pd.read_csv("data/raw_responses.csv")
errored = results_df[results_df["Error"].notna()]

print(f"Total rows:    {len(results_df)}")
print(f"Errored rows:  {len(errored)}")
print(f"Successful:    {len(results_df) - len(errored)}")
print()

print("=== Error breakdown by persona ===")
print(pd.crosstab(results_df["Persona"], results_df["Error"].notna(), 
                  rownames=["Persona"], colnames=["Errored"]))
print()

print("=== Error breakdown by Task_ID ===")
err_by_task = results_df.groupby("Task_ID")["Error"].apply(lambda x: x.notna().sum())
print(err_by_task.sort_values(ascending=False).head(10))
print()

print("=== Top 5 unique error messages ===")
print(errored["Error"].value_counts().head(5))
print()

print("=== Sample full error message ===")
if len(errored) > 0:
    print(repr(errored["Error"].iloc[0]))

Total rows:    7200
Errored rows:  2516
Successful:    4684

=== Error breakdown by persona ===
Errored                   False  True 
Persona                               
emory_grad                 1175    625
empty_nester               1143    657
skeptical_renter_control   1218    582
vahi_professional          1148    652

=== Error breakdown by Task_ID ===
Task_ID
35    200
34     89
4      74
12     70
19     70
17     70
11     70
3      69
1      69
27     69
Name: Error, dtype: int64

=== Top 5 unique error messages ===
Error
Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 50 requests per minute (org: 4829ef42-c67d-4625-b424-fb840ea31386, model: claude-haiku-4-5-20251001). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Please reduce the prompt length or the maximum tokens requested, or try again later. You m

In [16]:
### run again for those with errors

# === RETRY FAILED CALLS WITH CONSERVATIVE CONCURRENCY ===

# Load existing results
results_df = pd.read_csv("data/raw_responses.csv")
errored = results_df[results_df["Error"].notna()].copy()
successful = results_df[results_df["Error"].isna()].copy()

print(f"Total rows: {len(results_df)}")
print(f"To retry: {len(errored)}")
print(f"Already successful: {len(successful)}")

# Conservative settings for Tier 1 (50 RPM = 0.83/sec; we'll target ~40 RPM)
RETRY_CONCURRENCY = 3
RETRY_DELAY_SEC = 1.5  # extra delay between calls to stay under RPM

async def retry_one_call(client, semaphore, row, attr_names):
    """Retry a single failed call. row is a dict from errored.to_dict('records')."""
    task_id = row["Task_ID"]
    persona_name = row["Persona"]
    rep = row["Rep"]
    shuffled = json.loads(row["Shuffled_Profiles"])
    prompt = format_choice_prompt(shuffled, attr_names)

    async with semaphore:
        for attempt in range(5):  # more retries this time
            try:
                response = await client.messages.create(
                    model=FULL_MODEL,
                    max_tokens=120,
                    temperature=1.0,
                    system=PERSONAS[persona_name],
                    messages=[{"role": "user", "content": prompt}],
                )
                await asyncio.sleep(RETRY_DELAY_SEC)  # rate-limit pacing
                return {
                    "Task_ID": task_id,
                    "Persona": persona_name,
                    "Rep": rep,
                    "Shuffled_Profiles": row["Shuffled_Profiles"],
                    "Response": response.content[0].text,
                    "Error": None,
                }
            except Exception as e:
                if attempt == 4:
                    return {
                        "Task_ID": task_id,
                        "Persona": persona_name,
                        "Rep": rep,
                        "Shuffled_Profiles": row["Shuffled_Profiles"],
                        "Response": None,
                        "Error": str(e),
                    }
                # Longer backoff for rate limits: 5s, 10s, 20s, 40s
                await asyncio.sleep(5 * (2 ** attempt))
        return None

async def run_retries():
    async_client = anthropic.AsyncAnthropic()
    semaphore = asyncio.Semaphore(RETRY_CONCURRENCY)
    
    coros = [retry_one_call(async_client, semaphore, row, attr_names)
             for row in errored.to_dict("records")]
    
    print(f"Retrying {len(coros)} calls at concurrency={RETRY_CONCURRENCY}...")
    print(f"Estimated wall time: {len(coros) * RETRY_DELAY_SEC / RETRY_CONCURRENCY / 60:.1f} minutes")
    start = time.time()
    
    results = []
    chunk = 50
    for i in range(0, len(coros), chunk):
        batch = coros[i:i+chunk]
        batch_results = await asyncio.gather(*batch)
        results.extend(batch_results)
        elapsed = time.time() - start
        n_success = sum(1 for r in results if r and r.get("Error") is None)
        print(f"  {len(results)}/{len(coros)} done | {n_success} successful so far | elapsed {elapsed:.0f}s")
    
    return results

retry_results = await run_retries()

# Merge: keep originals that succeeded, replace errored with retry attempts
retry_df = pd.DataFrame(retry_results)
final_df = pd.concat([successful, retry_df], ignore_index=True)
final_df.to_csv("data/raw_responses.csv", index=False)

n_errors_final = final_df["Error"].notna().sum()
print(f"\n=== After retry ===")
print(f"Total: {len(final_df)}")
print(f"Errors remaining: {n_errors_final} ({100*n_errors_final/len(final_df):.1f}%)")

Total rows: 7200
To retry: 2516
Already successful: 4684
Retrying 2516 calls at concurrency=3...
Estimated wall time: 21.0 minutes
  50/2516 done | 50 successful so far | elapsed 52s
  100/2516 done | 100 successful so far | elapsed 102s
  150/2516 done | 150 successful so far | elapsed 153s
  200/2516 done | 200 successful so far | elapsed 206s
  250/2516 done | 250 successful so far | elapsed 256s
  300/2516 done | 300 successful so far | elapsed 317s
  350/2516 done | 350 successful so far | elapsed 376s
  400/2516 done | 400 successful so far | elapsed 436s
  450/2516 done | 450 successful so far | elapsed 502s
  500/2516 done | 500 successful so far | elapsed 556s
  550/2516 done | 550 successful so far | elapsed 617s
  600/2516 done | 600 successful so far | elapsed 677s
  650/2516 done | 650 successful so far | elapsed 739s
  700/2516 done | 700 successful so far | elapsed 798s
  750/2516 done | 750 successful so far | elapsed 858s
  800/2516 done | 800 successful so far | elaps

## Done — Move to Notebook 2

Outputs ready for analysis:
- `data/choice_tasks.csv` — the experimental design
- `data/raw_responses.csv` — Claude's choices across personas
- `data/choice_task_preview.html` — visual sample of the cards

Next: open `02_analyze_and_visualize.ipynb` for parsing → mixed logit → WTP → plots.